# Ingesting and Validating Data

In [1]:
import os
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition\\prototype'

In [2]:
os.chdir('../')
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition'

In [3]:
import io
import json
from minio import Minio
from minio.error import S3Error
from dotenv import load_dotenv
from include.common import read_yaml
from include.constants import CONFIG_FILE_PATH
import pandas as pd
from include.constants import EQ_COLUMNS
import numpy as np

load_dotenv()

True

# Reading in Data From MinIO

In [5]:
# Connection settings
endpoint = "localhost:9000"  # Update if using a different host/portq
access_key = os.getenv("MINIO_ACCESS_KEY", "minio")
secret_key = os.getenv("MINIO_SECRET_KEY", "minio123")

client = Minio(
    endpoint,
    access_key=access_key,
    secret_key=secret_key,
    secure=False,
)

BUCKET_NAME = "earthquake-raw"
OBJECT_NAME = "year=2026/month=03/day=21/raw.json"

response = client.get_object(BUCKET_NAME, OBJECT_NAME)
try:
    payload = json.loads(response.read().decode("utf-8"))
finally:
    response.close()
    response.release_conn()

payload['features']

[]

## Must flatten the feature key

In [32]:
# payload_path = "tests/usgs_eq/some_missing_geometry_key.json"

# with open(payload_path, "r", encoding="utf-8") as f:
#     payload = json.load(f)

# # len(payload['features'])

# df = pd.json_normalize(payload["features"])
# df['geometry.coordinates'].value_counts(dropna=False)
# len(df)


152

In [6]:
len(payload['features'])

0

In [7]:
payload

{'type': 'FeatureCollection',
 'metadata': {'generated': 1774053964000,
  'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-03-21T00%3A45%3A59&endtime=2026-03-22T00%3A45%3A59&minmagnitude=2.5&orderby=time-asc&limit=20000',
  'title': 'USGS Earthquakes',
  'status': 200,
  'api': '2.4.0',
  'limit': 20000,
  'offset': 1},
 'features': []}

In [8]:
# If some rows are JSON strings, parse them first.
def to_list(v) -> list:
    """
    Convert a JSON string to a list, or return the value as-is if it's not a string.
    """
    if isinstance(v, str):
        try:
            return json.loads(v)   # e.g. "[40.1, 9.2, 10.0]" -> [40.1, 9.2, 10.0]
        except Exception:
            return np.nan
    return v

In [14]:
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition'

In [11]:
c
coords = df["geometry.coordinates"].apply(to_list)

df["longitude"] = coords.str[0]
df["latitude"] = coords.str[1]
df["depth_km"] = coords.str[2]   # optional

df.drop(columns=["geometry.coordinates"], inplace=True)

# Remove only the properties prefix
df.columns

KeyError: 'geometry.coordinates'

In [ ]:
print(list(df.columns))

['type', 'id', 'properties.mag', 'properties.place', 'properties.time', 'properties.updated', 'properties.tz', 'properties.url', 'properties.detail', 'properties.felt', 'properties.cdi', 'properties.mmi', 'properties.alert', 'properties.status', 'properties.tsunami', 'properties.sig', 'properties.net', 'properties.code', 'properties.ids', 'properties.sources', 'properties.types', 'properties.nst', 'properties.dmin', 'properties.rms', 'properties.gap', 'properties.magType', 'properties.type', 'properties.title', 'geometry.type', 'geometry.coordinates']


: 

In [ ]:
len(df)

89

In [ ]:
df.isnull().sum()

type            0
id              0
mag             0
place           0
time            0
updated         0
tz             89
url             0
detail          0
felt           69
cdi            69
mmi            84
alert          88
status          0
tsunami         0
sig             0
net             0
code            0
ids             0
sources         0
types           0
nst             5
dmin            5
rms             0
gap             5
magType         0
type            0
title           0
type            0
coordinates     0
dtype: int64

In [ ]:
list(df.columns)

['type',
 'id',
 'mag',
 'place',
 'time',
 'updated',
 'tz',
 'url',
 'detail',
 'felt',
 'cdi',
 'mmi',
 'alert',
 'status',
 'tsunami',
 'sig',
 'net',
 'code',
 'ids',
 'sources',
 'types',
 'nst',
 'dmin',
 'rms',
 'gap',
 'magType',
 'type',
 'title',
 'type',
 'coordinates']